In [1]:
from sympy import Array, Symbol, tensorcontraction, tensorproduct
import black


def bending_function_sympy(Q, Bp, Bpp):
    QM = Q.reshape(4, 3).transpose()

    # Projections (3-vector each)

    yip = tensorcontraction(tensorproduct(QM, Bp), (1, 2))
    yipp = tensorcontraction(tensorproduct(QM, Bpp), (1, 2))

    def cross(a: Array, b: Array):
        """
        3d cross product
        https://reference.wolfram.com/language/ref/Cross.html
        """
        return Array([
            a[1] * b[2] - a[2] * b[1],
            a[2] * b[0] - a[0] * b[2],
            a[0] * b[1] - a[1] * b[0],
        ])

    def norm3(arr: Array):
        x, y, z = arr[0], arr[1], arr[2]
        return (x**2 + y**2 + z**2) ** (0.5)

    return (norm3(cross(yip, yipp)) / (norm3(yip) ** 3)) ** 2


sympy_res = bending_function_sympy(
    Array([Symbol(f"q{i}") for i in range(1, 13)]),
    Array([Symbol(f"bp{i}") for i in range(1, 5)]),
    Array([Symbol(f"bpp{i}") for i in range(1, 5)]),
)
print(black.format_str(repr(sympy_res), mode=black.Mode(line_length=88)).strip())

(
    (
        (bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10)
        * (bpp1 * q2 + bpp2 * q5 + bpp3 * q8 + bpp4 * q11)
        - (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11)
        * (bpp1 * q1 + bpp2 * q4 + bpp3 * q7 + bpp4 * q10)
    )
    ** 2
    + (
        -(bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10)
        * (bpp1 * q3 + bpp2 * q6 + bpp3 * q9 + bpp4 * q12)
        + (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12)
        * (bpp1 * q1 + bpp2 * q4 + bpp3 * q7 + bpp4 * q10)
    )
    ** 2
    + (
        (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11)
        * (bpp1 * q3 + bpp2 * q6 + bpp3 * q9 + bpp4 * q12)
        - (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12)
        * (bpp1 * q2 + bpp2 * q5 + bpp3 * q8 + bpp4 * q11)
    )
    ** 2
) ** 1.0 / (
    (bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10) ** 2
    + (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11) ** 2
    + (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12) ** 2
) ** 3.0


In [2]:
import numpy as np

# create random array inputs
ng = np.random.default_rng()
# control points
q_np = ng.random(12).astype(float)
# bernstein coefficients
bp_np = ng.random(4).astype(float)
bpp_np = ng.random(4).astype(float)

In [3]:
def bending_function(Q, Bp, Bpp):
    xp = Q.__array_namespace__()
    QM = xp.reshape(Q, (4, 3)).T

    yip = xp.vecdot(QM, Bp)
    yipp = xp.vecdot(QM, Bpp)
    num = xp.linalg.vector_norm(xp.cross(yip, yipp))
    den = xp.linalg.vector_norm(yip) ** 3
    return (num / den) ** 2


bending_function(q_np, bp_np, bpp_np)

# Try sympy on random inputs to make sure they agree
(
    bending_function_sympy(
        Array(q_np.tolist()),
        Array(bp_np.tolist()),
        Array(bpp_np.tolist()),
    ),
    bending_function(q_np, bp_np, bpp_np),
)

(0.000115197693947647, np.float64(0.0001151976939476468))

In [4]:
import egglog
import egglog.exp.array_api as enp

Bp = enp.NDArray([enp.Value.var(f"bp{i}") for i in range(1, 5)])
Bpp = enp.NDArray([enp.Value.var(f"bpp{i}") for i in range(1, 5)])
Q = enp.NDArray([enp.Value.var(f"q{i}") for i in range(1, 13)])
res = bending_function(Q, Bp, Bpp)
res

_NDArray_1 = reshape(
    NDArray(
        RecursiveValue.vec(
            Vec(
                RecursiveValue(Value.var("q1")),
                RecursiveValue(Value.var("q2")),
                RecursiveValue(Value.var("q3")),
                RecursiveValue(Value.var("q4")),
                RecursiveValue(Value.var("q5")),
                RecursiveValue(Value.var("q6")),
                RecursiveValue(Value.var("q7")),
                RecursiveValue(Value.var("q8")),
                RecursiveValue(Value.var("q9")),
                RecursiveValue(Value.var("q10")),
                RecursiveValue(Value.var("q11")),
                RecursiveValue(Value.var("q12")),
            )
        )
    ),
    TupleInt(Vec(Int(4), Int(3))),
).T
_NDArray_2 = vecdot(
    _NDArray_1,
    NDArray(
        RecursiveValue.vec(
            Vec(
                RecursiveValue(Value.var("bp1")),
                RecursiveValue(Value.var("bp2")),
                RecursiveValue(Value.var("bp3")),
                RecursiveValue(Value.var("bp4")),
            )
        )
    ),
)
(
    vector_norm(
        cross(
            _NDArray_2,
            vecdot(
                _NDArray_1,
                NDArray(
                    RecursiveValue.vec(
                        Vec(
                            RecursiveValue(Value.var("bpp1")),
                            RecursiveValue(Value.var("bpp2")),
                            RecursiveValue(Value.var("bpp3")),
                            RecursiveValue(Value.var("bpp4")),
                        )
                    )
                ),
            ),
        )
    )
    / vector_norm(_NDArray_2) ** NDArray(RecursiveValue(Value.from_int(Int(3))))
) ** NDArray(RecursiveValue(Value.from_int(Int(2))))

In [5]:
(scalar_res := res.eval())

_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from functools import total_ordering


@total_ordering
@dataclass(frozen=True)
class ArithmeticCost:
    """
    Cost type where *, +, and ** are the same and then / is more than all of them.
    """

    muls: int = 0
    other: int = 0
    adds: int = 0
    subs: int = 0
    exps: int = 0

    @property
    def not_div_ops(self) -> int:
        return self.muls + self.adds + self.exps + self.subs

    def __eq__(self, other: ArithmeticCost) -> bool:
        return self.not_div_ops == other.not_div_ops and self.other == other.other

    def __gt__(self, other: ArithmeticCost) -> bool:
        if self.other != other.other:
            return self.other > other.other
        return self.not_div_ops > other.not_div_ops

    def __add__(self, other: ArithmeticCost) -> ArithmeticCost:
        return ArithmeticCost(
            muls=self.muls + other.muls,
            other=self.other + other.other,
            adds=self.adds + other.adds,
            exps=self.exps + other.exps,
            subs=self.subs + other.subs,
        )

    def __sub__(self, other: ArithmeticCost) -> ArithmeticCost:
        return ArithmeticCost(
            muls=self.muls - other.muls,
            other=self.other - other.other,
            adds=self.adds - other.adds,
            exps=self.exps - other.exps,
            subs=self.subs - other.subs,
        )


def arith_cost_model(egraph: egglog.EGraph, expr: egglog.Expr, children_costs: list[ArithmeticCost]) -> ArithmeticCost:
    # literals are free
    if egglog.get_literal_value(expr) is not None:
        return ArithmeticCost()
    match egglog.get_callable_fn(expr):
        case enp.Value.__add__:
            c = ArithmeticCost(adds=1)
        case enp.Value.__sub__:
            c = ArithmeticCost(subs=1)
        case enp.Value.__mul__:
            c = ArithmeticCost(muls=1)
        case enp.Value.__pow__:
            c = ArithmeticCost(exps=1)
        case enp.Int | enp.Value.from_int:
            c = ArithmeticCost()
        case enp.Value.var | enp.NDArray | enp.RecursiveValue.vec | enp.RecursiveValue | egglog.Vec:
            c = ArithmeticCost()
        case _:
            c = ArithmeticCost(other=1)
    return sum(children_costs, c)


egraph = egglog.EGraph()
egraph.register(scalar_res)
_, original_cost = egraph.extract(scalar_res, include_cost=True, cost_model=arith_cost_model)
print(original_cost)
scalar_res

ArithmeticCost(muls=66, other=1, adds=49, subs=3, exps=7)


_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

Now let's try distributing...

In [7]:
@egglog.ruleset
def distribute(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.rewrite((a + b) * c, subsume=True).to(a * c + b * c)
    yield egglog.rewrite(c * (a + b), subsume=True).to(c * a + c * b)
    # Push down subtraction
    yield egglog.rewrite(a - b, subsume=True).to(a + (-1) * b)


egraph.run(distribute.saturate())
distributed_res, distributed_cost = egraph.extract(scalar_res, cost_model=arith_cost_model, include_cost=True)
print(distributed_cost)
distributed_res

ArithmeticCost(muls=348, other=1, adds=106, subs=0, exps=7)


_Value_1 = Value.var("q2") * Value.var("bp1")
_Value_2 = Value.var("q3") * Value.var("bpp1")
_Value_3 = Value.var("q6") * Value.var("bpp2")
_Value_4 = Value.var("q9") * Value.var("bpp3")
_Value_5 = Value.var("q12") * Value.var("bpp4")
_Value_6 = Value.var("q5") * Value.var("bp2")
_Value_7 = Value.var("q8") * Value.var("bp3")
_Value_8 = Value.var("q11") * Value.var("bp4")
_Value_9 = Value.from_int(Int(-1))
_Value_10 = Value.var("q3") * Value.var("bp1")
_Value_11 = Value.var("q2") * Value.var("bpp1")
_Value_12 = Value.var("q5") * Value.var("bpp2")
_Value_13 = Value.var("q8") * Value.var("bpp3")
_Value_14 = Value.var("q11") * Value.var("bpp4")
_Value_15 = Value.var("q6") * Value.var("bp2")
_Value_16 = Value.var("q9") * Value.var("bp3")
_Value_17 = Value.var("q12") * Value.var("bp4")
_Value_18 = Value.var("q1") * Value.var("bpp1")
_Value_19 = Value.var("q4") * Value.var("bpp2")
_Value_20 = Value.var("q7") * Value.var("bpp3")
_Value_21 = Value.var("q10") * Value.var("bpp4")
_Value_22 = Value.var("q1") * Value.var("bp1")
_Value_23 = Value.var("q4") * Value.var("bp2")
_Value_24 = Value.var("q7") * Value.var("bp3")
_Value_25 = Value.var("q10") * Value.var("bp4")
(
    (
        _Value_1 * _Value_2
        + _Value_1 * _Value_3
        + _Value_1 * _Value_4
        + _Value_1 * _Value_5
        + (
            _Value_6 * _Value_2
            + _Value_6 * _Value_3
            + _Value_6 * _Value_4
            + _Value_6 * _Value_5
        )
        + (
            _Value_7 * _Value_2
            + _Value_7 * _Value_3
            + _Value_7 * _Value_4
            + _Value_7 * _Value_5
        )
        + (
            _Value_8 * _Value_2
            + _Value_8 * _Value_3
            + _Value_8 * _Value_4
            + _Value_8 * _Value_5
        )
        + (
            _Value_9 * (_Value_10 * _Value_11)
            + _Value_9 * (_Value_10 * _Value_12)
            + _Value_9 * (_Value_10 * _Value_13)
            + _Value_9 * (_Value_10 * _Value_14)
            + (
                _Value_9 * (_Value_15 * _Value_11)
                + _Value_9 * (_Value_15 * _Value_12)
                + _Value_9 * (_Value_15 * _Value_13)
                + _Value_9 * (_Value_15 * _Value_14)
            )
            + (
                _Value_9 * (_Value_16 * _Value_11)
                + _Value_9 * (_Value_16 * _Value_12)
                + _Value_9 * (_Value_16 * _Value_13)
                + _Value_9 * (_Value_16 * _Value_14)
            )
            + (
                _Value_9 * (_Value_17 * _Value_11)
                + _Value_9 * (_Value_17 * _Value_12)
                + _Value_9 * (_Value_17 * _Value_13)
                + _Value_9 * (_Value_17 * _Value_14)
            )
        )
    )
    ** Value.from_int(Int(2))
    + (
        _Value_10 * _Value_18
        + _Value_10 * _Value_19
        + _Value_10 * _Value_20
        + _Value_10 * _Value_21
        + (
            _Value_15 * _Value_18
            + _Value_15 * _Value_19
            + _Value_15 * _Value_20
            + _Value_15 * _Value_21
        )
        + (
            _Value_16 * _Value_18
            + _Value_16 * _Value_19
            + _Value_16 * _Value_20
            + _Value_16 * _Value_21
        )
        + (
            _Value_17 * _Value_18
            + _Value_17 * _Value_19
            + _Value_17 * _Value_20
            + _Value_17 * _Value_21
        )
        + (
            _Value_9 * (_Value_22 * _Value_2)
            + _Value_9 * (_Value_22 * _Value_3)
            + _Value_9 * (_Value_22 * _Value_4)
            + _Value_9 * (_Value_22 * _Value_5)
            + (
                _Value_9 * (_Value_23 * _Value_2)
                + _Value_9 * (_Value_23 * _Value_3)
                + _Value_9 * (_Value_23 * _Value_4)
                + _Value_9 * (_Value_23 * _Value_5)
            )
            + (
                _Value_9 * (_Value_24 * _Value_2)
                + _Value_9 * (_Value_24 * _Value_3)
                + _Value_9 * (_Value_24 * _Value_4)
                + _Value_9 

In [8]:
egraph = egglog.EGraph()
egraph.register(distributed_res)
egraph.run(enp.polynomial_schedule)
factored_res, factored_cost = egraph.extract(distributed_res, include_cost=True, cost_model=arith_cost_model)
print(factored_cost)
factored_res

ArithmeticCost(muls=159, other=1, adds=106, subs=0, exps=7)


_Value_1 = (
    Value.var("bp1") * Value.var("q2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("bp2") * Value.var("q5")
    + Value.var("bp4") * Value.var("q11")
)
_Value_2 = (
    Value.var("q12") * Value.var("bp4")
    + Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("bp3") * Value.var("q9")
)
_Value_3 = (
    Value.var("bp1") * Value.var("q1")
    + Value.var("bp3") * Value.var("q7")
    + Value.var("bp2") * Value.var("q4")
    + Value.var("bp4") * Value.var("q10")
)
(
    (
        Value.var("bpp4") * (Value.var("q12") * _Value_1)
        + Value.from_int(Int(-1))
        * (
            Value.var("bpp4") * (Value.var("q11") * _Value_2)
            + Value.var("q5") * (Value.var("bpp2") * _Value_2)
            + Value.var("q2") * (Value.var("bpp1") * _Value_2)
            + Value.var("bpp3") * (Value.var("q8") * _Value_2)
        )
        + Value.var("q9") * (Value.var("bpp3") * _Value_1)
        + Value.var("bpp2") * (Value.var("q6") * _Value_1)
        + Value.var("bpp1") * (Value.var("q3") * _Value_1)
    )
    ** Value.from_int(Int(2))
    + (
        Value.var("bpp4") * (Value.var("q10") * _Value_2)
        + Value.from_int(Int(-1))
        * (
            Value.var("bpp4") * (Value.var("q12") * _Value_3)
            + Value.var("q9") * (Value.var("bpp3") * _Value_3)
            + Value.var("bpp2") * (Value.var("q6") * _Value_3)
            + Value.var("bpp1") * (Value.var("q3") * _Value_3)
        )
        + Value.var("q4") * (Value.var("bpp2") * _Value_2)
        + Value.var("q7") * (Value.var("bpp3") * _Value_2)
        + Value.var("q1") * (Value.var("bpp1") * _Value_2)
    )
    ** Value.from_int(Int(2))
    + (
        Value.var("bpp4") * (Value.var("q11") * _Value_3)
        + Value.from_int(Int(-1))
        * (
            Value.var("bpp4") * (Value.var("q10") * _Value_1)
            + Value.var("q4") * (Value.var("bpp2") * _Value_1)
            + Value.var("q7") * (Value.var("bpp3") * _Value_1)
            + Value.var("q1") * (Value.var("bpp1") * _Value_1)
        )
        + Value.var("q5") * (Value.var("bpp2") * _Value_3)
        + Value.var("q2") * (Value.var("bpp1") * _Value_3)
        + Value.var("bpp3") * (Value.var("q8") * _Value_3)
    )
    ** Value.from_int(Int(2))
) / (
    _Value_3 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_2 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [9]:
original_cost.not_div_ops

125

In [10]:
factored_cost.not_div_ops

272

In [11]:
distributed_cost.not_div_ops

461

In [12]:
@egglog.ruleset
def factoring(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.birewrite((a + b) * c).to(a * c + b * c)
    yield egglog.rewrite(a * b).to(b * a)
    yield egglog.rewrite(a + b).to(b + a)


egraph = egglog.EGraph()
egraph.register(distributed_res)
egraph.run(factoring.saturate())
factored_ac_result, factored_ac_cost = egraph.extract(distributed_res, cost_model=arith_cost_model, include_cost=True)
print(factored_ac_cost)
factored_ac_result

ArithmeticCost(muls=69, other=1, adds=52, subs=0, exps=7)


_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 + _Value_3 * _Value_4 * Value.from_int(Int(-1)))
    ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 + _Value_6 * _Value_2 * Value.from_int(Int(-1)))
    ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 + _Value_1 * _Value_5 * Value.from_int(Int(-1)))
    ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [13]:
egraph.all_function_sizes()

[(Int, 3),
 (· + ·, 806),
 (· * ·, 930),
 (· ** ·, 7),
 (· / ·, 1),
 (Value.from_int, 3),
 (Value.var, 20)]

In [14]:
factored_ac_cost.not_div_ops

128

In [15]:
scalar_res

_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [16]:
original_cost.not_div_ops

125

In [17]:
(FunctionBending := res.eval())

_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [18]:
egraph = egglog.EGraph()
GradientBending = enp.NDArray(FunctionBending).diff(Q)
egraph.register(GradientBending)
egraph.run(enp.array_api_schedule)
GradientBending = egraph.extract(GradientBending)

egraph = egglog.EGraph()
egraph.register(GradientBending)
egraph.run(enp.array_api_schedule)
GradientBending = egraph.extract(GradientBending)

egraph = egglog.EGraph()
egraph.register(GradientBending)
(GradientBending, GradientBending_cost) = egraph.extract(GradientBending, include_cost=True)

GradientBending_cost

19994

In [19]:
@egglog.ruleset
def distribute_pow(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.rewrite((a + b) * c, subsume=True).to(a * c + b * c)
    yield egglog.rewrite(c * (a + b), subsume=True).to(c * a + c * b)
    # Push down subtraction
    yield egglog.rewrite(a - b, subsume=True).to(a + (-1) * b)

    # yield egglog.rewrite(a ** 2, subsume=True).to(a * a)
    # yield egglog.rewrite(a ** 3, subsume=True).to(a * a * a)


egraph = egglog.EGraph()
egraph.register(GradientBending)
egraph.run(distribute_pow.saturate())
(GradientBending_distributed, GradientBending_distributed_cost) = egraph.extract(GradientBending, include_cost=True)
GradientBending_distributed_cost

4250786

In [20]:
import time

egraph = egglog.EGraph()
GradientBending_distributed_let = egraph.let("$GradientBending_distributed", GradientBending_distributed)
start = time.perf_counter()
sizes = [egraph.all_function_sizes()]
for ruleset in [enp.to_polynomial_ruleset, enp.factor_ruleset, enp.from_polynomial_ruleset]:
    egraph.run(ruleset.saturate())
    sizes.append(egraph.all_function_sizes())
stop = time.perf_counter()
run_time = stop - start
print(run_time)


GradientBending_horner_factored, GradientBending_horner_factored_cost = egraph.extract(
    GradientBending_distributed_let, include_cost=True
)
GradientBending_horner_factored_cost

4.612682083999971


162182

In [26]:
sum(size for _, size in sizes[-1])

100608

In [22]:
import time

egraph = egglog.EGraph()
egraph.register(GradientBending_distributed)
collected = []
for i in range(20):
    n_nodes = sum(size for _, size in egraph.all_function_sizes())
    start = time.perf_counter()
    egraph.run(factoring)
    stop = time.perf_counter()
    run_time = stop - start
    print(i, run_time)

    _, GradientBending_factored_cost = egraph.extract(GradientBending_distributed, include_cost=True)
    collected.append((n_nodes, run_time, GradientBending_factored_cost))

0 0.04176395799731836
1 0.4749334160005674
2 0.92753920800169
3 1.0339599589933641
4 0.408640750014456
5 0.36490983300609514
6 0.3627474159875419
7 0.23935820799670182
8 0.20117004198255017
9 0.557841957983328
10 0.9708038749813568
11 1.6268839580006897
12 2.125054417003412
13 2.498730708990479
14 2.865033583017066
15 4.0420821250008885
16 4.123069624998607
17 4.461457875004271
18 4.830629082978703
19 4.788934000011068


In [23]:
collected

[(56289, 0.04176395799731836, 3974202),
 (62170, 0.4749334160005674, 3427186),
 (81248, 0.92753920800169, 2195164),
 (101214, 1.0339599589933641, 1315601),
 (113617, 0.408640750014456, 852292),
 (121547, 0.36490983300609514, 496246),
 (128231, 0.3627474159875419, 338817),
 (133653, 0.23935820799670182, 187592),
 (141106, 0.20117004198255017, 137387),
 (153683, 0.557841957983328, 93438),
 (174803, 0.9708038749813568, 67160),
 (202729, 1.6268839580006897, 43520),
 (232819, 2.125054417003412, 32477),
 (272738, 2.498730708990479, 23691),
 (322046, 2.865033583017066, 18758),
 (378278, 4.0420821250008885, 16610),
 (437774, 4.123069624998607, 15970),
 (493556, 4.461457875004271, 15754),
 (549894, 4.830629082978703, 15614),
 (609735, 4.788934000011068, 15614)]

In [24]:
# GradientBending_horner_factored

In [ ]:
# frozen = egraph.freeze()
# root = frozen.global_bindings['$GradientBending_distributed']

KeyboardInterrupt: 

In [ ]:
# egglog.get_callable_args(egglog.get_callable_args(frozen.outputs[egglog.get_callable_args(frozen.outputs[root][0].output)[0]][0].output)[0])

(Value(7283),
 Value(11742),
 Value(16198),
 Value(20658),
 Value(25114),
 Value(29571),
 Value(34028),
 Value(38483),
 Value(42938),
 Value(47386),
 Value(51832),
 Value(56286))

In [ ]:
# @egglog.ruleset
# def non_terminate(x: enp.Value, y: enp.Value, z: enp.Value):
#     yield egglog.rewrite((x * y) * z).to(x * (y * z))
#     print(egglog.rewrite(0 * x).to(x))
#     yield egglog.rewrite(0 * x).to(x)

# init = 0 * egglog.constant("a", enp.Value)
# print(init)
# egraph = egglog.EGraph(save_egglog_string=True)
# egraph.register(init)
# egraph.run(non_terminate * 5)
# print(egraph.as_egglog_string)
# egraph.saturate(non_terminate, max=5)a

In [ ]:
# x1, x2, x3, x4, x5, x6, x7, x8, x9, x10 = (egglog.constant(f"x{i}", enp.Value) for i in range(1, 11))

# (EXPR := (

# (x1 + x2 / 2 - x3/3 + x4/4)**6 *
#        (x5 + 2*x6 - 3*x7 + 4*x8)**5 * (x9 + x10)**2
#      - (x1 + x2/4 - x3/3 + x4/2)**7 *
#        (x5 + 4*x6 - 3*x7 + 2*x8)**4 * (x9 - x10)**2
# ))

In [ ]:
# @egglog.ruleset
# def remove_div_exp(x: enp.Value, y: enp.Value, i: egglog.i64):
#     # Replace div with multiplcation
#     yield egglog.rewrite(x / i, subsume=True).to(x * egglog.BigRat(1, i))
#     # Replace power with multiplcation
#     yield egglog.rewrite(x ** i, subsume=True).to(x ** (i -1) * x, i != 1)
#     yield egglog.rewrite(x ** 1, subsume=True).to(x)

# egraph = egglog.EGraph()
# egraph.register(EXPR)
# egraph.run(remove_div_exp.saturate())
# egraph.extract(EXPR)

In [ ]:
# egraph.run(distribute.saturate())
# egraph.extract(EXPR)